# Portofolio Data Science - Pertemuan 10
- **Nama Lengkap**: Muhammad Ikctiar Saputra
- **NIM**: 250401020169
- **Kelas**: IF401
- **Program Studi**: PJJ Informatika

---

## Langkah 1: Mengunduh dan Membaca Dataset Telco Customer Churn

In [ ]:
import urllib.request
import pandas as pd

# Mengunduh dataset Telco Churn dari repositori IBM
url_dataset = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
file_lokal = "telco_churn_data.csv"
urllib.request.urlretrieve(url_dataset, file_lokal)

# Memuat dataset ke dalam DataFrame
df_churn = pd.read_csv(file_lokal)
print("Ukuran Dataset:", df_churn.shape)
print(df_churn.head())

## Langkah 2: Pembersihan Data dan Pembagian Dataset

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Menampilkan tipe data kolom
print(df_churn.info())

# Membersihkan kolom TotalCharges (terdapat string kosong)
df_churn['TotalCharges'] = pd.to_numeric(df_churn['TotalCharges'], errors='coerce')
median_charges = df_churn['TotalCharges'].median()
df_churn['TotalCharges'] = df_churn['TotalCharges'].fillna(median_charges)

# Menghapus kolom customerID karena tidak relevan untuk pemodelan
df_churn = df_churn.drop(columns=['customerID'])

# Mengubah kolom target Churn menjadi numerik
df_churn['Churn'] = df_churn['Churn'].map({'Yes': 1, 'No': 0})

# Memisahkan fitur dan target
X_fitur = df_churn.drop(columns=['Churn'])
y_target = df_churn['Churn']

# Encoding variabel kategori menggunakan One-Hot Encoding
X_encoded = pd.get_dummies(X_fitur, drop_first=True)

# Membagi data menjadi train & test set (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_target, test_size=0.2, stratify=y_target, random_state=42
)

print("Dimensi data latih:", X_train.shape)
print("Dimensi data uji:", X_test.shape)
print("Distribusi kelas target Churn (prosentase):")
print(y_target.value_counts(normalize=True) * 100)

## Langkah 3: Pelatihan Model Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Inisialisasi model Random Forest dengan penyeimbang bobot kelas
rf_classifier = RandomForestClassifier(
    n_estimators=300,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42
)

# Pelatihan model
rf_classifier.fit(X_train, y_train)
y_pred = rf_classifier.predict(X_test)

## Langkah 4: Evaluasi Performa Model dan Analisis Fitur Terpenting

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt

# Menampilkan metrik evaluasi klasifikasi
print("=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))
print(f"ROC AUC Score: {roc_auc_score(y_test, rf_classifier.predict_proba(X_test)[:, 1]):.4f}")

# Menampilkan Fitur Terpenting
importances = rf_classifier.feature_importances_
df_importances = pd.DataFrame({
    'Fitur': X_encoded.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Visualisasi 10 Fitur Terpenting
plt.figure(figsize=(12, 6))
plt.barh(df_importances['Fitur'].head(10)[::-1], df_importances['Importance'].head(10)[::-1], color='steelblue')
plt.xlabel('Tingkat Kepentingan Fitur')
plt.title('10 Fitur Paling Berpengaruh dalam Prediksi Churn')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

## Kesimpulan & Pembahasan

Berdasarkan pemodelan Random Forest Classifier untuk memprediksi churn pelanggan:
1. **Analisis Metrik Evaluasi**: Dikarenakan data target yang tidak seimbang (sekitar 73.4% No dan 26.6% Yes), penggunaan parameter `class_weight='balanced'` sangat membantu meningkatkan nilai *Recall* untuk kelas Churn (1). Hal ini penting karena kegagalan mendeteksi pelanggan yang akan churn berakibat kerugian bisnis yang lebih besar bagi perusahaan.
2. **Pilar Utama Penentu Churn**: Berdasarkan visualisasi tingkat kepentingan fitur (*feature importance*), variabel kuantitatif seperti `TotalCharges`, `MonthlyCharges`, dan `tenure` menjadi faktor paling signifikan. Pelanggan dengan tenure (durasi langganan) yang pendek dan biaya bulanan yang tinggi cenderung memiliki risiko churn yang lebih tinggi.
3. **Rekomendasi Bisnis**: Perusahaan perlu memberikan insentif khusus atau promo retensi bagi pelanggan baru (tenure rendah) yang memiliki tagihan bulanan tinggi agar menekan laju churn.